In [1]:
import re, sys, os
from pathlib import Path
import torch
import numpy as np
repo_start = f'../'
sys.path.append(repo_start)

from modules.utils.imports import *
from modules.binn_eql.model_wrapper_2d import model_wrapper
from modules.binn_eql.build_binn_eql_net import BINN
# from modules.binn_eql_hypernet.build_binn_eql_net import BINN
from modules.utils.format_data import format_u_array_to_training_data

In [2]:
def extract_final_loss(filepath, loss_types=['pde', 'gls', 'reg']):
    """
    Reads the given file and extracts the validation loss (as a float)
    from the line containing the word 'Elapsed'.
    """
    # Regular expression to capture the validation loss
    # This regex looks for "Val loss = " followed by a floating point number (possibly in scientific notation)
    # val_loss_pattern = re.compile(r"Val loss = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    pde_pattern = re.compile(r"Val PDE = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    gls_pattern = re.compile(r"Val GLS = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
    reg_pattern = re.compile(r"Val Reg = ([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)")
        
    with open(filepath, 'r') as file:
        for line in file:
            # Check if the line contains "Elapsed"
            if "Elapsed" in line:
                # Search the line for the pattern
                pde = float(pde_pattern.search(line).group(1))
                gls = float(gls_pattern.search(line).group(1))
                reg = float(reg_pattern.search(line).group(1))
                
                loss = 0
                
                if 'pde' in loss_types: loss += pde
                if 'gls' in loss_types: loss += gls
                if 'reg' in loss_types: loss += reg
                
                return loss
                
    # If no matching line was found
    return None

def process_directory(parent_dir, loss_types):
    """
    For each subdirectory under parent_dir, finds slurm files with the pattern "slurm-####.out",
    selects the one with the lower number, extracts the validation loss, and prints the subdirectory path
    alongside the extracted loss.
    """
    results = {}
    # Use pathlib to iterate over subdirectories
    for subdir in Path(parent_dir).iterdir():
        if subdir.is_dir():
            # Find files matching pattern "slurm-*.out"
            slurm_files = list(subdir.glob("slurm-*.out"))
            if not slurm_files:
                # print(f"No slurm files found in {subdir}")
                results[str(subdir)] = None
                continue
            
            # Extract numeric part from filename (assumes naming: slurm-####.out)
            def extract_number(file_path):
                match = re.search(r"slurm-(\d+)\.out", file_path.name)
                return int(match.group(1)) if match else float('inf')
            
            # Sort files by the extracted number
            slurm_files.sort(key=extract_number)
            # Take the first file (lowest number)
            first_file = slurm_files[0]
            val_loss = extract_final_loss(first_file, loss_types)
            
            if val_loss is not None:
                results[str(subdir)] = val_loss
            else:
                # print(f"Could not extract validation loss from file {first_file} in {subdir}")
                results[str(subdir)] = None
    
    return results

training_data_path = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/data/wave_pinning/positive_feedback/du_0.01_dv_1.0_a_1.0_b_1.0_k_0.01.pt'
training_data = torch.load(training_data_path)['training_data']

In [ ]:
parent_directory = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql/runs/pos_feedback/79_uv_unscaled/du_0.01_dv_1.0_a_5.0_b_1.0_k_0.01'
loss_types = ['reg', 'pde']

losses = process_directory(parent_directory, loss_types)

# Sort the results by validation loss (lowest first)
sorted_losses = sorted(losses.items(), key=lambda item: (item[1] is None, item[1]))

# Print the sorted results
for dir_name, val_loss in sorted_losses:
    if val_loss:
        print(f"\nDirectory: {dir_name.split('/')[-1]}")
        print(f"Validation Loss: {val_loss}")
        
        # Load model
        binn = BINN(
            dimensions=2,
            species=2, 
            train_data=training_data, 
            diff_coeffs=[0.01, 1],
            # uv_layers=[256, 256, 256, 256, 2],
            # diff_coeffs=(),
            duplicates=5)
            # duplicates=20)

        binn.to('cpu')

        parameters = binn.parameters()

        opt = torch.optim.Adam(parameters, lr=0.001)

        model = model_wrapper(
            model=binn,
            optimizer=opt,
            loss=binn.loss,
            dir_name=dir_name,
            save_name=f'{dir_name}/binn')
                
        model.load(f"{dir_name}/binn_best_val_model", device='cpu')
        # model.model.prune(thresh=5)
        # Print equation
        fn = f'{dir_name}/equation.txt'
        for term in model.model.generate_equation():
            print(f'{term}')
        
        model.model.fine_tune_eql(threshold=0.01, epsilon=0.1)
        for term in model.model.generate_equation():
            print(f'{term}')
            
        if not model.model.diff_coeffs:          
            print(f'{[D.item() for D in model.model.diffusion_fitter()]}\n')



Directory: binn_eql_duplicates_5_repeat_9
Validation Loss: 2.051465
-1.300 * u^1.021 / (1 + 0.501 * u^1.021)
4.008 * v * u^1.934 / (1 + 0.046 * u^1.934)
Simplifying Hill 41 -> Poly 0
  Error: 0.0694, Multiplier: 0.7529
Fine-tuning committed.
-0.979 * u
4.008 * v * u^1.934 / (1 + 0.046 * u^1.934)

Directory: binn_eql_duplicates_5_repeat_1
Validation Loss: 2.058706
-0.784 * u
3.686 * v * u^2.131 / (1 + 0.014 * u^2.131)
Fine-tuning committed.
-0.784 * u
3.686 * v * u^2.131 / (1 + 0.014 * u^2.131)

Directory: binn_eql_duplicates_5_repeat_4
Validation Loss: 2.058793
3.719 * v * u^2.055 / (1 + 0.038 * u^2.055)
-1.097 * u^1.000 / (1 + 0.304 * u^1.000)
Simplifying Hill 57 -> Poly 0
  Error: 0.0504, Multiplier: 0.8155
Fine-tuning committed.
-0.895 * u
3.719 * v * u^2.055 / (1 + 0.038 * u^2.055)

Directory: binn_eql_duplicates_5_repeat_8
Validation Loss: 2.060941
-1.279 * u^1.014 / (1 + 0.482 * u^1.014)
4.048 * v * u^1.912 / (1 + 0.045 * u^1.912)
Simplifying Hill 25 -> Poly 0
  Error: 0.0693, M

In [32]:
# Load model
binn = BINN(
    dimensions=2,
    species=2, 
    train_data=training_data, 
    diff_coeffs=[0.01, 1],
    # uv_layers=[256, 256, 256, 256, 2],
    # diff_coeffs=(),
    duplicates=5)

binn.to('cpu')

parameters = binn.parameters()

opt = torch.optim.Adam(parameters, lr=0.001)

model = model_wrapper(
    model=binn,
    optimizer=opt,
    loss=binn.loss,
    dir_name=dir_name,
    save_name=f'{dir_name}/binn')

model.load(f"/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql_net/runs/pos_feedback/76_more_l0_loss/a_1.0_b_1_k_0.01/binn_eql_l0_2_duplicates_5_warm_up_20000_lux_tax_1_repeat_3/binn_best_val_model", device='cpu')

# DIFFUSION COEFFICIENTS

In [7]:
parent_directory = '/work/users/s/m/smyersn/elston/projects/kinetics_binns/development/binn_eql/runs/diff_coeffs/9_d_lim_0.01/du_0.01_dv_5.0_a_1.0_b_1.0_k_0.01'
loss_types = ['pde']

losses = process_directory(parent_directory, loss_types)

# Sort the results by validation loss (lowest first)
sorted_losses = sorted(losses.items(), key=lambda item: (item[1] is None, item[1]))

# Print the sorted results
for dir_name, val_loss in sorted_losses:
    if val_loss:
        print(f"\nDirectory: {dir_name.split('/')[-1]}")
        print(f"Validation Loss: {val_loss}")
        
        # Load model
        binn = BINN(
            dimensions=2,
            species=2, 
            train_data=training_data, 
            diff_coeffs=[],
            # uv_layers=[256, 256, 256, 256, 2],
            # diff_coeffs=(),
            duplicates=5)
            # duplicates=20)

        binn.to('cpu')

        parameters = binn.parameters()

        opt = torch.optim.Adam(parameters, lr=0.001)

        model = model_wrapper(
            model=binn,
            optimizer=opt,
            loss=binn.loss,
            dir_name=dir_name,
            save_name=f'{dir_name}/binn')
                
        model.load(f"{dir_name}/binn_best_val_model", device='cpu')
        # model.model.prune(thresh=5)
        # Print equation
        fn = f'{dir_name}/equation.txt'
        for term in model.model.generate_equation():
            print(f'{term}')
        
        model.model.fine_tune_eql(threshold=0.01, epsilon=0.1)
        for term in model.model.generate_equation():
            print(f'{term}')
            
        if not model.model.diff_coeffs:          
            print(f'{[D.item() for D in model.model.diffusion_fitter()]}\n')



Directory: binn_eql_duplicates_5_repeat_3
Validation Loss: 0.025107
0.773 * v * u^2.080 / (1 + 0.009 * u^2.080)
-0.244 * u * (1 / 0.258 - v^1.000 / (1 + 0.258 * v^1.000))
Simplifying Hill 64 -> Poly 0
  Error: 0.0669, Multiplier: 3.4984
Fine-tuning committed.
-0.852 * u
0.773 * v * u^2.080 / (1 + 0.009 * u^2.080)
[0.009999999776482582, 4.527677059173584]


Directory: binn_eql_duplicates_5_repeat_1
Validation Loss: 0.025235
0.568 * v * (1 / 0.005 - u^1.226 / (1 + 0.005 * u^1.226))
-0.309 * u * (1 / 0.334 - v^1.413 / (1 + 0.334 * v^1.413))
-1.235 * v * (1 / 0.011 - u^1.902 / (1 + 0.011 * u^1.902))
Merging Duplicate Hills: 39 and 63 (Dist: 0.0000)
Simplifying Hill 39 -> Poly 1
  Error: 0.0014, Multiplier: 5421.4600
Simplifying Hill 56 -> Poly 0
  Error: 0.0865, Multiplier: 2.7149
Fine-tuning committed.
-0.838 * u
-3612.887 * v
[0.009999999776482582, 4.536374568939209]


Directory: binn_eql_duplicates_5_repeat_2
Validation Loss: 0.033649
-0.924 * u
0.928 * v * u^1.993 / (1 + 0.010 * u^1.9